# Traffic Data Visualization

This notebook visualizes traffic speed data from the Traffic API using MapboxGL.

In [1]:
# Install required packages
!pip install mapboxgl jupyter pandas requests -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
import json

## Configuration

In [3]:
# API endpoint
API_BASE = "http://localhost:8000"

# Mapbox token - REPLACE WITH YOUR TOKEN
MAPBOX_TOKEN = "YOUR_MAPBOX_TOKEN_HERE"

## Fetch Data from API

In [4]:
def fetch_aggregates(day, period):
    """Fetch aggregated speed data for a day and period."""
    url = f"{API_BASE}/aggregates/"
    params = {"day": day, "period": period}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


def fetch_spatial_filter(day, period, bbox):
    """Fetch road segments within a bounding box."""
    url = f"{API_BASE}/aggregates/spatial_filter/"
    payload = {"day": day, "period": period, "bbox": bbox}
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()


def fetch_slow_links(period, threshold, min_days):
    """Fetch slow links based on threshold."""
    url = f"{API_BASE}/patterns/slow_links/"
    params = {"period": period, "threshold": threshold, "min_days": min_days}
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

## Test API Endpoints

In [5]:
# Test aggregates endpoint
print("Testing /aggregates/ endpoint...")
aggregates = fetch_aggregates("Tuesday", "AM Peak")
print(f"Retrieved {len(aggregates.get('data', []))} links")
print("Sample:", aggregates.get("data", [{}])[0] if aggregates.get("data") else "No data")

Testing /aggregates/ endpoint...
Retrieved 5 links
Sample: {'link_id': '1240632857', 'avg_speed': 40.34, 'name': 'E 21st St'}


In [6]:
# Test spatial filter endpoint
print("Testing /aggregates/spatial_filter/ endpoint...")
bbox = [-81.8, 30.1, -81.6, 30.3]
spatial = fetch_spatial_filter("Tuesday", "AM Peak", bbox)
print(f"Retrieved {len(spatial.get('data', []))} segments in bbox {bbox}")
print("Sample:", spatial.get("data", [{}])[0] if spatial.get("data") else "No data")

Testing /aggregates/spatial_filter/ endpoint...
Retrieved 5 segments in bbox [-81.8, 30.1, -81.6, 30.3]
Sample: {'link_id': '1240474884', 'avg_speed': 37.84, 'name': 'University Blvd W', 'geometry': '{"type":"MultiLineString","coordinates":[[[-81.60999,30.27097],[-81.60958,30.27139]]]}'}


In [7]:
# Test slow links endpoint
print("Testing /patterns/slow_links/ endpoint...")
slow_links = fetch_slow_links("PM Peak", 30.0, 3)
print(f"Retrieved {len(slow_links.get('data', []))} slow links")
print("Sample:", slow_links.get("data", [{}])[0] if slow_links.get("data") else "No data")

Testing /patterns/slow_links/ endpoint...


Retrieved 0 slow links
Sample: No data


## Visualize with MapboxGL

## Visualize with Folium

MapboxGL requires a valid token and has API changes. Using Folium for visualization.

In [8]:
# Get spatial data with geometry
spatial_response = fetch_spatial_filter("Tuesday", "AM Peak", [-81.8, 30.1, -81.6, 30.3])
spatial_data = spatial_response.get("data", [])
print(f"Got {len(spatial_data)} road segments")

Got 5 road segments


In [9]:
# Convert to GeoJSON for visualization
geojson_features = []
for item in spatial_data:
    if item.get("geometry"):
        feature = {
            "type": "Feature",
            "properties": {
                "link_id": item["link_id"],
                "name": item.get("name", "Unknown"),
                "avg_speed": item["avg_speed"],
            },
            "geometry": json.loads(item["geometry"]),
        }
        geojson_features.append(feature)

geojson_collection = {"type": "FeatureCollection", "features": geojson_features}

print(f"Created GeoJSON with {len(geojson_features)} features")

Created GeoJSON with 5 features


## Alternative: Simple Visualization with Folium

In [10]:
# Install folium for alternative visualization
!pip install folium -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
import folium

# Create a simple map
m2 = folium.Map(location=[30.2, -81.7], zoom_start=10, tiles="cartodbpositron")

# Add geojson layer
folium.GeoJson(
    geojson_collection,
    name="traffic_speed",
    style_function=lambda x: {
        "color": "red"
        if x["properties"]["avg_speed"] < 20
        else "orange"
        if x["properties"]["avg_speed"] < 40
        else "green"
        if x["properties"]["avg_speed"] >= 60
        else "yellow",
        "weight": 3,
    },
).add_to(m2)

folium.LayerControl().add_to(m2)
m2

## Summary

This notebook demonstrates:
1. Fetching data from the Traffic API
2. Visualizing road segments with speed data using MapboxGL
3. Alternative visualization with Folium

To use MapboxGL properly, replace `YOUR_MAPBOX_TOKEN_HERE` with your actual Mapbox access token.